<a href="https://colab.research.google.com/github/vishnuTeja8/-Predictive-Restaurant-Recommender/blob/main/assignment%5Bsoulpage%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predictive Restaurant Recommender — Colab Notebook
Author: Vishnu Teja


In [1]:
!pip install -q lightgbm==3.3.5

import os
import gc
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 3) Load files -- adapt filenames if necessary
# Typical filenames (adjust if your files have other names)
data_dir = '/content/' # Updated data directory
fn_train_customers = os.path.join(data_dir, 'train_customers.csv')
fn_train_locations = os.path.join(data_dir, 'train_locations.csv')
fn_train_orders    = os.path.join(data_dir, 'orders.csv') # Note: The actual file is 'orders.csv' based on the listing
fn_vendors         = os.path.join(data_dir, 'vendors.csv')
fn_test_customers  = os.path.join(data_dir, 'test_customers.csv')
fn_test_locations  = os.path.join(data_dir, 'test_locations.csv')

# helper to load while checking alternatives
def load_best(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    # try matching by partial name
    base = os.path.basename(path).split('.')[0]
    for f in os.listdir(data_dir):
        if base in f:
            return pd.read_csv(os.path.join(data_dir, f))
    raise FileNotFoundError(f'{path} not found in {data_dir}')

print("Loading files (this may take a while)...")
train_cust = load_best(fn_train_customers)
train_loc  = load_best(fn_train_locations)
train_ord  = load_best(fn_train_orders)
vendors    = load_best(fn_vendors)
test_cust  = load_best(fn_test_customers)
test_loc   = load_best(fn_test_locations)

print("Shapes:")
print("train_customers", train_cust.shape)
print("train_locations", train_loc.shape)
print("train_orders", train_ord.shape)
print("vendors", vendors.shape)
print("test_customers", test_cust.shape)
print("test_locations", test_loc.shape)

Loading files (this may take a while)...
Shapes:
train_customers (34674, 8)
train_locations (59503, 5)
train_orders (41764, 26)
vendors (100, 59)
test_customers (9768, 8)
test_locations (16720, 5)


/tmp/ipython-input-3497979771.py:14: DtypeWarning: Columns (15,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


In [4]:
import os
print(os.listdir('/content/'))

['.config', 'test_locations.csv', 'train_customers.csv', 'drive', 'train_locations.csv', 'vendors.csv', 'orders.csv', 'test_customers.csv', 'sample_data']


In [5]:
# 4) Quick EDA snapshots
def quick_eda(df, name, n=5):
    print(f'--- {name} --- shape: {df.shape}')
    display(df.head(n))
    # Only sample if there are enough columns, otherwise display all
    if df.describe(include='all').T.shape[0] >= 10:
        display(df.describe(include='all').T.sample(10))
    else:
        display(df.describe(include='all').T)

quick_eda(train_cust, 'train_customers')
quick_eda(train_loc, 'train_locations')
quick_eda(train_ord, 'train_orders')
quick_eda(vendors, 'vendors')
quick_eda(test_cust, 'test_customers')
quick_eda(test_loc, 'test_locations')

--- train_customers --- shape: (34674, 8)


,customer_id,gender,dob,status,verified,language,created_at,updated_at
0,TCHWPBT,Male,NaN,1,1,EN,2/7/2023 19:16,2/7/2023 19:16
1,ZGFSYCZ,Male,NaN,1,1,EN,2/9/2023 12:04,2/9/2023 12:04
2,S2ALZFL,Male,NaN,0,1,EN,3/14/2023 18:31,3/14/2023 18:31
3,952DBJQ,Male,NaN,1,1,EN,3/15/2023 19:47,3/15/2023 19:47
4,1IX6FXS,Male,NaN,1,1,EN,3/15/2023 19:57,3/15/2023 19:57


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,34674,34523,0FOCFVI,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,22520,10,Male,17815,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dob,3046.0,NaN,NaN,NaN,1991.210768,48.422045,1.0,1986.0,1993.0,1999.0,2562.0
status,34674.0,NaN,NaN,NaN,0.998991,0.031756,0.0,1.0,1.0,1.0,1.0
verified,34674.0,NaN,NaN,NaN,0.956538,0.203898,0.0,1.0,1.0,1.0,1.0
language,21099,1,EN,21099,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,34674,30242,10/9/2024 6:41,226,NaN,NaN,NaN,NaN,NaN,NaN,NaN
updated_at,34674,22319,10/1/2024 18:50,503,NaN,NaN,NaN,NaN,NaN,NaN,NaN


--- train_locations --- shape: (59503, 5)


,customer_id,location_number,location_type,latitude,longitude
0,02SFNJH,0,NaN,1.682392,-78.789737
1,02SFNJH,1,NaN,1.679137,0.766823
2,02SFNJH,2,NaN,-0.498648,0.661241
3,RU43CXC,0,Home,0.100853,0.438165
4,BDFBPRD,0,NaN,2.523125,0.733464


...

,customer_id,gender,dob,status,verified,language,created_at,updated_at
0,ICE2DJP,Male,NaN,1,1,EN,2/7/2023 16:45,2/7/2023 16:45
1,FWNUI71,Male,NaN,1,1,EN,3/22/2023 20:11,3/22/2023 20:11
2,LRX7BCH,Male,NaN,1,1,EN,4/17/2023 20:01,4/17/2023 20:01
3,D96DHMD,Male,NaN,1,1,EN,4/29/2023 22:35,4/29/2023 22:35
4,88Q8Y5V,Male,1997.0,1,1,EN,5/5/2023 12:38,5/5/2023 12:38


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,9768,9753,JGRG3RQ,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,6321,6,Male,5021,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dob,848.0,NaN,NaN,NaN,1993.246462,23.844723,1900.0,1987.0,1994.0,1999.0,2562.0
status,9768.0,NaN,NaN,NaN,0.998976,0.031981,0.0,1.0,1.0,1.0,1.0
verified,9768.0,NaN,NaN,NaN,0.955569,0.206061,0.0,1.0,1.0,1.0,1.0
language,5928,1,EN,5928,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,9768,9162,10/9/2024 6:41,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN
updated_at,9768,6874,10/1/2024 18:50,141,NaN,NaN,NaN,NaN,NaN,NaN,NaN


--- test_locations --- shape: (16720, 5)


,customer_id,location_number,location_type,latitude,longitude
0,Z59FTQD,0,NaN,126.032278,-9.106019
1,0JP29SK,0,Home,0.278709,-78.623847
2,0JP29SK,1,Home,0.124485,-78.605621
3,0JP29SK,2,NaN,-0.113891,-78.577449
4,0JP29SK,3,NaN,-0.848796,0.136726


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,16720,10000,T1L5K29,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_number,16720.0,NaN,NaN,NaN,0.721112,1.163847,0.0,0.0,0.0,1.0,11.0
location_type,9070,3,Home,5504,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,16717.0,NaN,NaN,NaN,2.51485,22.105672,-671.693058,-0.513176,-0.098033,0.246075,453.653846
longitude,16717.0,NaN,NaN,NaN,-25.270783,36.937377,-132.204545,-78.52129,0.011745,0.509318,44.121945


In [20]:
# 5) Basic cleaning & type conversion
# Convert datetime columns in orders if present
date_cols = [c for c in train_ord.columns if 'date' in c.lower() or 'created_at' in c.lower() or 'delivery_date' in c.lower()]
for c in date_cols:
    try:
        train_ord[c] = pd.to_datetime(train_ord[c])
    except Exception as e:
        print(f'Could not parse {c}: {e}')
# example: compute order_date if available
if 'created_at' in train_ord.columns:
    train_ord['order_date'] = pd.to_datetime(train_ord['created_at'])
elif 'delivery_date' in train_ord.columns:
    train_ord['order_date'] = pd.to_datetime(train_ord['delivery_date'])
else:
    train_ord['order_date'] = pd.NaT


In [8]:
# 5) Basic cleaning & type conversion (moved from cell eMZ1lHbaoclZ)
# Convert datetime columns in orders if present
date_cols = [c for c in train_ord.columns if 'date' in c.lower() or 'created_at' in c.lower() or 'delivery_date' in c.lower()]
for c in date_cols:
    try:
        train_ord[c] = pd.to_datetime(train_ord[c])
    except Exception as e:
        print(f'Could not parse {c}: {e}')
# example: compute order_date if available
if 'created_at' in train_ord.columns:
    train_ord['order_date'] = pd.to_datetime(train_ord['created_at'])
elif 'delivery_date' in train_ord.columns:
    train_ord['order_date'] = pd.to_datetime(train_ord['delivery_date'])
else:
    train_ord['order_date'] = pd.NaT


# 6) Feature engineering - vendor popularity and customer aggregations
# Vendor popularity (total orders per vendor)
vendor_pop = train_ord.groupby('vendor_id').size().rename('vendor_orders').reset_index()
vendors = vendors.merge(vendor_pop, left_on='id', right_on='vendor_id', how='left')

# Fix: Access the correct column after merge and rename, then drop redundant columns
vendors['vendor_orders'] = vendors['vendor_orders_y'].fillna(0).astype(int)
vendors = vendors.drop(columns=['vendor_id_x', 'vendor_orders_x', 'vendor_id_y', 'vendor_orders_y'], errors='ignore')


# Customer-level aggregates
cust_agg = train_ord.groupby('customer_id').agg(
    customer_total_orders = ('order_id','count'),
    customer_avg_grand_total = ('grand_total','mean'),
    customer_last_order = ('order_date','max')
).reset_index()
train_cust = train_cust.merge(cust_agg, on='customer_id', how='left')

# Customer-vendor order counts (useful to detect repeat vendors)
cust_vendor = train_ord.groupby(['customer_id','vendor_id']).size().rename('cust_vendor_orders').reset_index()

print("Vendor popularity sample")
display(vendors[['id','vendor_orders']].sort_values('vendor_orders', ascending=False).head())

print("Customer-vendor sample")
display(cust_vendor.head())

Vendor popularity sample


,id,vendor_orders
13,78,3880
27,113,3180
17,83,2922
19,85,1670
44,193,1446


Customer-vendor sample


,customer_id,vendor_id,cust_vendor_orders
0,002510Y,157.0,1
1,009UFS1,83.0,1
2,009UFS1,193.0,1
3,00GV4J4,189.0,1
4,00HWUU3,85.0,1


In [9]:
# 7) Create training pairs (customer-location-vendor) with label=1 (positive examples)
# We assume location number column is 'LOCATION_NUMBER' or 'location_number'
loc_col = 'LOCATION_NUMBER' if 'LOCATION_NUMBER' in train_ord.columns else 'location_number'
train_pos = train_ord[[ 'customer_id', loc_col, 'vendor_id', 'order_date' ]].drop_duplicates()
train_pos = train_pos.rename(columns={loc_col:'location_number', 'vendor_id':'vendor_id'})
train_pos['label'] = 1
train_pos = train_pos[['customer_id','location_number','vendor_id','order_date','label']]

print("Positive pairs:", train_pos.shape)
train_pos.head()


Positive pairs: (41645, 5)


,customer_id,location_number,vendor_id,order_date,label
0,KL09J9N,0.0,84.0,2024-08-02 05:33:00,1
1,H5LGGFX,0.0,78.0,2024-08-02 05:34:00,1
2,CYLZB6T,0.0,4.0,2024-08-02 05:35:00,1
3,4YKUKYN,0.0,157.0,2024-08-02 05:39:00,1
4,WDNU30K,0.0,160.0,2024-08-02 05:39:00,1


In [10]:
# 8) Negative sampling: for each positive pair sample k negative vendors not in that customer's vendor list
# Build list of all vendor ids
all_vendor_ids = sorted(vendors['id'].unique().tolist())

# Map customer -> vendors ordered
cust2vendors = cust_vendor.groupby('customer_id')['vendor_id'].apply(set).to_dict()

# Negative sampling function
import random
random.seed(42)
def sample_negatives_for_row(row, k=4):
    c = row['customer_id']
    positive_vendors = cust2vendors.get(c, set())
    candidates = [v for v in all_vendor_ids if v not in positive_vendors]
    if len(candidates) == 0:
        return []
    sample = random.sample(candidates, min(k, len(candidates)))
    return [(c, row['location_number'], v) for v in sample]

# Generate negatives
neg_samples = []
k_neg = 4
for idx,row in train_pos.iterrows():
    negs = sample_negatives_for_row(row, k=k_neg)
    for c,loc,v in negs:
        neg_samples.append((c,loc,v))
len(neg_samples)


166580

In [11]:
# convert negatives to DataFrame
train_neg = pd.DataFrame(neg_samples, columns=['customer_id','location_number','vendor_id'])
train_neg['order_date'] = pd.NaT
train_neg['label'] = 0

# Combine train_pos and train_neg
train_pairs = pd.concat([train_pos[['customer_id','location_number','vendor_id','order_date','label']], train_neg], ignore_index=True)
print("Total training pairs:", train_pairs.shape)
train_pairs.head()


Total training pairs: (208225, 5)


,customer_id,location_number,vendor_id,order_date,label
0,KL09J9N,0.0,84.0,2024-08-02 05:33:00,1
1,H5LGGFX,0.0,78.0,2024-08-02 05:34:00,1
2,CYLZB6T,0.0,4.0,2024-08-02 05:35:00,1
3,4YKUKYN,0.0,157.0,2024-08-02 05:39:00,1
4,WDNU30K,0.0,160.0,2024-08-02 05:39:00,1


In [12]:
# 9) Feature construction for pairs
# We'll construct:
# - vendor_orders (popularity)
# - vendor_rating (if available)
# - distance between customer location and vendor (requires joining customer_location lat/lon with vendor lat/lon)
# - whether customer ever ordered from vendor previously (binary)
# - customer_total_orders
# - recency (days since last order to pair order_date) -- for training only
# Note: train_loc and test_loc contain masked lat/lon per location

# Standardize column names in location tables
train_loc = train_loc.rename(columns=lambda x: x.strip())
test_loc  = test_loc.rename(columns=lambda x: x.strip())

# Build customer-location table (unique customer-location rows)
custloc = train_loc.rename(columns={'customer_id':'customer_id','location_number':'location_number'})
# ensure lat/lon columns present
custloc_cols = [c for c in custloc.columns if 'lat' in c.lower() or 'long' in c.lower()]
print("custloc columns", custloc.columns)

# Merge pair-level features - select necessary columns directly from vendors
pairs = train_pairs.merge(custloc[['customer_id','location_number'] + custloc_cols], on=['customer_id','location_number'], how='left')
pairs = pairs.merge(vendors[['id','latitude','longitude','vendor_orders','vendor_rating']].rename(columns={'id':'vendor_id', 'latitude':'vendor_latitude', 'longitude':'vendor_longitude'}), on='vendor_id', how='left')


# Feature: was this vendor previously ordered by customer?
pairs['ever_ordered'] = pairs.apply(lambda r: 1 if r['vendor_id'] in cust2vendors.get(r['customer_id'], set()) else 0, axis=1)

# Feature: distance (euclidean on masked coords) - if lat/lon available
if len(custloc_cols) >= 2 and 'vendor_latitude' in pairs.columns and 'vendor_longitude' in pairs.columns:
    # find likely names
    # try: 'Latitude' & 'longitude' or similar
    lat_col = [c for c in custloc_cols if 'lat' in c.lower()][0]
    lon_col = [c for c in custloc_cols if 'long' in c.lower() or 'lon' in c.lower()][0]
    def safe_dist(r):
        try:
            return np.sqrt((r[lat_col] - r['vendor_latitude'])**2 + (r[lon_col] - r['vendor_longitude'])**2)
        except:
            return np.nan
    pairs['dist'] = pairs.apply(safe_dist, axis=1)
else:
    pairs['dist'] = np.nan

# Merge customer_total_orders, customer_avg_grand_total, customer_last_order directly from train_cust
pairs = pairs.merge(train_cust[['customer_id','customer_total_orders','customer_avg_grand_total','customer_last_order']], on='customer_id', how='left')

# recency in days for rows that have order_date
pairs['recency_days'] = (pd.to_datetime(pairs['order_date']) - pd.to_datetime(pairs['customer_last_order'])).dt.days
pairs['recency_days'] = pairs['recency_days'].fillna(99999)

# Fill NaNs
pairs['vendor_orders'] = pairs['vendor_orders'].fillna(0)
pairs['vendor_rating'] = pairs['vendor_rating'].fillna(pairs['vendor_rating'].median())
pairs['customer_total_orders'] = pairs['customer_total_orders'].fillna(0)
pairs['customer_avg_grand_total'] = pairs['customer_avg_grand_total'].fillna(0)
pairs['dist'] = pairs['dist'].fillna(pairs['dist'].median())

feature_cols = ['vendor_orders','vendor_rating','ever_ordered','customer_total_orders','customer_avg_grand_total','dist','recency_days']
pairs[feature_cols + ['label']].head()

custloc columns Index(['customer_id', 'location_number', 'location_type', 'latitude',
       'longitude'],
      dtype='object')


,vendor_orders,vendor_rating,ever_ordered,customer_total_orders,customer_avg_grand_total,dist,recency_days,label
0,44.0,4.3,1,2.0,10.100000,78.664246,0.0,1
1,3880.0,4.4,1,6.0,7.433333,79.025335,-25.0,1
2,917.0,4.4,1,1.0,15.000000,79.302094,0.0,1
3,750.0,4.3,1,3.0,28.733333,79.175514,-14.0,1
4,134.0,4.3,1,4.0,10.600000,78.589365,-21.0,1


In [13]:
# 10) Train / validation split and LightGBM training
X = pairs[feature_cols]
y = pairs['label'].astype(int)

# simple split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.1,
    'num_leaves': 31,
    'seed': 42,
    'verbosity': -1
}

bst = lgb.train(params, train_data, valid_sets=[train_data, valid_data], num_boost_round=1000, early_stopping_rounds=50, verbose_eval=50)
# Evaluate
val_pred = bst.predict(X_val, num_iteration=bst.best_iteration)
print("Val AUC:", roc_auc_score(y_val, val_pred))
print("Val AP:", average_precision_score(y_val, val_pred))


/usr/local/lib/python3.12/dist-packages/lightgbm/engine.py:181: UserWarning: 'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. Pass 'early_stopping()' callback via 'callbacks' argument instead.
  _log_warning("'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. "
/usr/local/lib/python3.12/dist-packages/lightgbm/engine.py:239: UserWarning: 'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. "


Training until validation scores don't improve for 50 rounds
[50]	training's auc: 1	valid_1's auc: 0.999939
Early stopping, best iteration is:
[1]	training's auc: 1	valid_1's auc: 0.99994
Val AUC: 0.9999399687837676
Val AP: 0.9999039500540281


In [14]:
# 11) Prepare test pairs (customer-location-vendor) for prediction.
# The required submission format expects a row per (CID, LOC_NUM, VENDOR).
# We'll create all combinations: num_test_customers * num_vendors (should be manageable: ~10k * 100)
# NOTE: this can be large in memory; if memory issues, generate per-chunk.

test_loc_cols = [c for c in test_loc.columns if 'lat' in c.lower() or 'long' in c.lower()]
test_custloc = test_loc[['customer_id','location_number'] + test_loc_cols].drop_duplicates()

# Build a cross join between test_custloc and all vendors
test_custloc['key'] = 1
vendor_meta2 = vendors[['id','latitude','longitude','vendor_orders','vendor_rating']].rename(columns={'id':'vendor_id', 'latitude':'vendor_latitude', 'longitude':'vendor_longitude'}).copy()
vendor_meta2['key'] = 1
test_pairs = test_custloc.merge(vendor_meta2, on='key', how='outer').drop(columns=['key'])

# features for test pairs (similar to training)
# was previously ordered: for test customers probably false; but if a test customer also exists in training, check
# Calculate 'ever_ordered' by merging with cust_vendor
test_pairs = test_pairs.merge(cust_vendor[['customer_id', 'vendor_id', 'cust_vendor_orders']], on=['customer_id', 'vendor_id'], how='left')
test_pairs['ever_ordered'] = test_pairs['cust_vendor_orders'].apply(lambda x: 1 if pd.notna(x) else 0)
test_pairs = test_pairs.drop(columns=['cust_vendor_orders'])


# distance
if len(test_loc_cols) >= 2:
    lat_col = [c for c in test_loc_cols if 'lat' in c.lower()][0]
    lon_col = [c for c in test_loc_cols if 'long' in c.lower() or 'lon' in c.lower()][0]
    def safe_dist_test(r):
        try:
            return np.sqrt((r[lat_col] - r['vendor_latitude'])**2 + (r[lon_col] - r['vendor_longitude'])**2)
        except:
            return np.nan
    test_pairs['dist'] = test_pairs.apply(safe_dist_test, axis=1)
else:
    test_pairs['dist'] = np.nan

# merge customer aggregates if exist in train
test_pairs = test_pairs.merge(train_cust[['customer_id','customer_total_orders','customer_avg_grand_total','customer_last_order']], on='customer_id', how='left')
test_pairs['customer_total_orders'] = test_pairs['customer_total_orders'].fillna(0)
test_pairs['customer_avg_grand_total'] = test_pairs['customer_avg_grand_total'].fillna(0)
test_pairs['recency_days'] = 99999
test_pairs['vendor_orders'] = test_pairs['vendor_orders'].fillna(0)
test_pairs['vendor_rating'] = test_pairs['vendor_rating'].fillna(test_pairs['vendor_rating'].median())
test_pairs['dist'] = test_pairs['dist'].fillna(test_pairs['dist'].median())

test_X = test_pairs[feature_cols]
print("Test pairs shape:", test_pairs.shape)
test_pairs.head()

Test pairs shape: (1672000, 15)


,customer_id,location_number,latitude,longitude,vendor_id,vendor_latitude,vendor_longitude,vendor_orders,vendor_rating,ever_ordered,dist,customer_total_orders,customer_avg_grand_total,customer_last_order,recency_days
0,Z59FTQD,0,126.032278,-9.106019,4,-0.588596,0.754434,917,4.4,0,127.004229,0.0,0.0,NaT,99999
1,Z59FTQD,0,126.032278,-9.106019,13,-0.471654,0.744470,319,4.7,0,126.886866,0.0,0.0,NaT,99999
2,Z59FTQD,0,126.032278,-9.106019,20,-0.407527,0.643681,736,4.5,0,126.815144,0.0,0.0,NaT,99999
3,Z59FTQD,0,126.032278,-9.106019,23,-0.585385,0.753811,361,4.5,0,127.000979,0.0,0.0,NaT,99999
4,Z59FTQD,0,126.032278,-9.106019,28,0.480602,0.552850,394,4.4,0,125.922663,0.0,0.0,NaT,99999


In [16]:
# 12) Predict probabilities for test pairs and create submission
test_pairs['pred_prob'] = bst.predict(test_X, num_iteration=bst.best_iteration)

# Strategy to create final target:
# Option A: mark top K vendors per customer-location as predicted = 1 (common approach)
# Option B: use threshold like 0.5 to mark positives
# We'll do top-3 vendors per (customer_id, location_number) as '1' by default (you can change K)

K = 3

def mark_topk(group):
    group = group.sort_values('pred_prob', ascending=False)
    group['target'] = 0
    topk = group.head(K).index
    group.loc[topk, 'target'] = 1
    return group

submission_df = test_pairs.groupby(['customer_id','location_number'], group_keys=False).apply(mark_topk)
submission_df = submission_df.reset_index(drop=True)

# final required columns: "CID X LOC_NUM X VENDOR target"
# We'll output columns in that exact order (CID, LOC_NUM, VENDOR, target)
submission = submission_df.rename(columns={'customer_id':'CID','location_number':'LOC_NUM','vendor_id':'VENDOR'})[['CID','LOC_NUM','VENDOR','target','pred_prob']]
submission.head(20)


/tmp/ipython-input-1950287628.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  submission_df = test_pairs.groupby(['customer_id','location_number'], group_keys=False).apply(mark_topk)


,CID,LOC_NUM,VENDOR,target,pred_prob
0,Z59FTQD,0,4,1,0.180747
1,Z59FTQD,0,13,1,0.180747
2,Z59FTQD,0,20,1,0.180747
3,Z59FTQD,0,23,0,0.180747
4,Z59FTQD,0,28,0,0.180747
5,Z59FTQD,0,33,0,0.180747
6,Z59FTQD,0,43,0,0.180747
7,Z59FTQD,0,44,0,0.180747
8,Z59FTQD,0,55,0,0.180747
9,Z59FTQD,0,66,0,0.180747


In [18]:
# Format to match "CID X LOC_NUM X VENDOR target"
submission_final = pd.DataFrame()
submission_final['CID X LOC_NUM X VENDOR'] = (
    submission_df['customer_id'].astype(str) + " X " +
    submission_df['location_number'].astype(str) + " X " +
    submission_df['vendor_id'].astype(str)
)
submission_final['target'] = submission_df['target']

# Save in correct format
out_path = os.path.join(data_dir, 'submission_final.csv')
submission_final.to_csv(out_path, index=False)
print("Saved final submission to:", out_path)

submission_final.head(20)

Saved final submission to: /content/submission_final.csv


,CID X LOC_NUM X VENDOR,target
0,Z59FTQD X 0 X 4,1
1,Z59FTQD X 0 X 13,1
2,Z59FTQD X 0 X 20,1
3,Z59FTQD X 0 X 23,0
4,Z59FTQD X 0 X 28,0
5,Z59FTQD X 0 X 33,0
6,Z59FTQD X 0 X 43,0
7,Z59FTQD X 0 X 44,0
8,Z59FTQD X 0 X 55,0
9,Z59FTQD X 0 X 66,0
